# Linked List Deep Dive

Linked lists are fundamental linear data structures where elements are stored in nodes, each pointing to the next. Understanding their trade-offs vs arrays is crucial for algorithm design and many LeetCode problems.

---

## Part 1: Theory

### What is a Linked List?

A **linked list** consists of **nodes**, where each node contains:
- **Data** (the value)
- **Next pointer** (reference to the next node)

```
Node: [value|next] → Node: [value|next] → ... → Node: [value|None]
```

### Types of Linked Lists

| Type | Description | Next | Prev |
|------|-------------|------|------|
| **Singly** | Each node points to next | ✓ | ✗ |
| **Doubly** | Each node points to next and prev | ✓ | ✓ |
| **Circular** | Last node points back to first | ✓ | ✗ (or ✓) |

### Linked List vs Array

| Operation | Array | Linked List | When to use |
|-----------|-------|-------------|-------------|
| **Access by index** | O(1) | O(n) | Array: frequent random access |
| **Insert at front** | O(n) | O(1) | Linked List: frequent front insertions |
| **Delete at front** | O(n) | O(1) | Linked List: frequent front deletions |
| **Insert at middle** | O(n) | O(n) | Similar (need traversal) |
| **Delete at middle** | O(n) | O(n) | Similar (need traversal) |
| **Insert at end** | O(1) amortized | O(n) | Array: frequent appends |
| **Delete at end** | O(1) | O(n) | Array: frequent pops |
| **Memory overhead** | Minimal | Extra pointer per node | Array: memory-constrained |
| **Cache locality** | Excellent | Poor | Array: CPU cache matters |

### Pros & Cons

**Pros of Linked Lists:**
- ✅ Dynamic size — no need to pre-allocate
- ✅ O(1) insertion/deletion at head (and tail for doubly with tail pointer)
- ✅ Easy to implement stack/queue behavior
- ✅ Memory efficient for variable-sized elements

**Cons of Linked Lists:**
- ❌ O(n) random access — can't do `list[i]` quickly
- ❌ Extra memory for pointers
- ❌ Poor cache locality — scattered memory
- ❌ Harder to debug (pointer chasing)
- ❌ No binary search (can't jump to middle)

### When to Choose Linked List

✅ **Use linked list when:**
- Frequent insertions/deletions at the beginning
- Implementing stack/queue
- Memory allocation overhead is acceptable
- Unknown size at allocation time

❌ **Use array when:**
- Need fast random access
- Memory is tight
- Cache performance matters
- Size is known or grows predictably

---

## Part 2: Python Implementation

In [ ]:
# --- Singly Linked List Node ---
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    
    def __repr__(self):
        return f"ListNode({self.val})"

# --- Singly Linked List Class ---
class LinkedList:
    def __init__(self):
        self.head = None
    
    def append(self, val):
        """Add to end"""
        if not self.head:
            self.head = ListNode(val)
            return
        curr = self.head
        while curr.next:
            curr = curr.next
        curr.next = ListNode(val)
    
    def prepend(self, val):
        """Add to beginning"""
        self.head = ListNode(val, self.head)
    
    def __repr__(self):
        nodes = []
        curr = self.head
        while curr:
            nodes.append(str(curr.val))
            curr = curr.next
        return " -> ".join(nodes) + " -> None"

# --- Test ---
ll = LinkedList()
ll.append(1)
ll.append(2)
ll.append(3)
print(ll)  # 1 -> 2 -> 3 -> None

ll.prepend(0)
print(ll)  # 0 -> 1 -> 2 -> 3 -> None

In [ ]:
# --- Doubly Linked List Node ---
class DListNode:
    def __init__(self, val=0, prev=None, next=None):
        self.val = val
        self.prev = prev
        self.next = next

# --- Doubly Linked List Class ---
class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
    
    def append(self, val):
        """Add to end - O(1) with tail pointer"""
        new_node = DListNode(val)
        if not self.head:
            self.head = self.tail = new_node
        else:
            self.tail.next = new_node
            new_node.prev = self.tail
            self.tail = new_node
    
    def prepend(self, val):
        """Add to beginning - O(1)"""
        new_node = DListNode(val)
        if not self.head:
            self.head = self.tail = new_node
        else:
            self.head.prev = new_node
            new_node.next = self.head
            self.head = new_node
    
    def __repr__(self):
        nodes = []
        curr = self.head
        while curr:
            nodes.append(str(curr.val))
            curr = curr.next
        return " <-> ".join(nodes) + " <-> None"

# --- Test ---
dll = DoublyLinkedList()
dll.append(1)
dll.append(2)
dll.append(3)
print(dll)  # 1 <-> 2 <-> 3 <-> None

dll.prepend(0)
print(dll)  # 0 <-> 1 <-> 2 <-> 3 <-> None

---

## Part 3: Common LeetCode Patterns

### Pattern 1 — Two Pointers (Fast/Slow)

**Concept**: Use two pointers moving at different speeds through the list.

**Why it works**: 
- **Fast pointer (2 steps)** explores ahead while **slow pointer (1 step)** trails behind
- When fast reaches the end, slow is exactly at the middle (for odd length) or first middle (for even length)
- For cycles: fast will eventually "lap" slow if a cycle exists (like runners on a track)

**Common use cases**:
- Find middle node (LeetCode 876)
- Detect cycles (LeetCode 141, 142)
- Find intersection point of two lists (LeetCode 160)

**Key insight**: You can get "two positions for the price of one" traversal — perfect when you need to know both current and future states simultaneously.

In [ ]:
# Find middle of linked list (LeetCode 876)
def middle_node(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow  # slow is at middle

# Detect cycle (LeetCode 141)
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast:
            return True
    return False

# --- Test ---
# Create list: 1 -> 2 -> 3 -> 4 -> 5
n1, n2, n3, n4, n5 = ListNode(1), ListNode(2), ListNode(3), ListNode(4), ListNode(5)
n1.next, n2.next, n3.next, n4.next = n2, n3, n4, n5
print(f"Middle: {middle_node(n1).val}")  # 3

# Create cycle: 1 -> 2 -> 3 -> 2 (cycle)
c1, c2, c3 = ListNode(1), ListNode(2), ListNode(3)
c1.next, c2.next, c3.next = c2, c3, c2
print(f"Has cycle: {has_cycle(c1)}")  # True

### Pattern 2 — Dummy Head

**Concept**: Insert a dummy node before the actual head to simplify edge cases.

**Why it works**:
- **Uniform handling**: Head becomes just another node — no special cases for head removal
- **Consistent API**: Always return `dummy.next` as the new head
- **Simpler logic**: No need to check if `prev` exists before setting `prev.next`

**Common use cases**:
- Remove head node (LeetCode 203)
- Merge lists (LeetCode 21)
- Insert at beginning
- Any operation that might change the head

**Key insight**: The dummy node absorbs all edge cases, letting you write cleaner code without nested `if` checks for head operations.

In [ ]:
# Remove elements with given value (LeetCode 203)
def remove_elements(head, val):
    dummy = ListNode(0, head)  # dummy.next = head
    prev = dummy
    curr = head
    
    while curr:
        if curr.val == val:
            prev.next = curr.next  # skip curr
        else:
            prev = curr  # move prev only when not removing
        curr = curr.next
    
    return dummy.next  # new head

# --- Test ---
# Create: 1 -> 2 -> 6 -> 3 -> 4 -> 5 -> 6
h = ListNode(1, ListNode(2, ListNode(6, ListNode(3, ListNode(4, ListNode(5, ListNode(6)))))))
new_head = remove_elements(h, 6)
# Should be: 1 -> 2 -> 3 -> 4 -> 5
result = []
while new_head:
    result.append(new_head.val)
    new_head = new_head.next
print(result)  # [1, 2, 3, 4, 5]

### Pattern 3 — Reversal

**Concept**: Reverse the direction of `next` pointers while traversing.

**Why it works**:
- **In-place**: No extra space needed — just rewire existing connections
- **Three-step dance**: Store next → reverse current → move to next
- **Iterative**: O(n) time, O(1) space
- **Recursive**: Elegant but uses O(n) call stack

**Common use cases**:
- Reverse entire list (LeetCode 206)
- Reverse sublist (LeetCode 92)
- Check palindrome (reverse + compare)
- Rotate list (reverse segments)

**Key insight**: Reversal is just changing the direction of arrows. Always keep track of the previous node because you lose the original `next` when you rewire.

In [ ]:
# Reverse entire list (LeetCode 206)
def reverse_list(head):
    prev = None
    curr = head
    while curr:
        nxt = curr.next  # store next
        curr.next = prev  # reverse pointer
        prev = curr  # move prev
        curr = nxt  # move curr
    return prev  # new head

# Reverse between positions (LeetCode 92)
def reverse_between(head, left, right):
    dummy = ListNode(0, head)
    prev = dummy
    
    # Move prev to node before left
    for _ in range(left - 1):
        prev = prev.next
    
    # Reverse sublist
    curr = prev.next
    for _ in range(right - left):
        nxt = curr.next
        curr.next = nxt.next
        nxt.next = prev.next
        prev.next = nxt
    
    return dummy.next

# --- Test ---
# Create: 1 -> 2 -> 3 -> 4 -> 5
h = ListNode(1, ListNode(2, ListNode(3, ListNode(4, ListNode(5)))))
rev = reverse_list(h)
result = []
while rev:
    result.append(rev.val)
    rev = rev.next
print(f"Reversed: {result}")  # [5, 4, 3, 2, 1]

### Pattern 4 — Merge

**Concept**: Combine two sorted lists by always advancing the smaller node.

**Why it works**:
- **Sorted property**: At each step, the smaller of two heads is the next in merged order
- **Linear time**: Each node is visited exactly once
- **Stable**: Maintains original order for equal elements
- **Recursive elegance**: Natural fit for divide-and-conquer

**Common use cases**:
- Merge two sorted lists (LeetCode 21)
- Merge k sorted lists (LeetCode 23)
- Add two numbers (LeetCode 2)
- Sort using merge sort

**Key insight**: Merging is like zipping two sorted arrays — always pick the smallest available element and advance its pointer.

In [ ]:
# Merge two sorted lists (LeetCode 21)
def merge_two_lists(l1, l2):
    dummy = ListNode()
    curr = dummy
    
    while l1 and l2:
        if l1.val <= l2.val:
            curr.next = l1
            l1 = l1.next
        else:
            curr.next = l2
            l2 = l2.next
        curr = curr.next
    
    # Append remaining
    curr.next = l1 if l1 else l2
    return dummy.next

# --- Test ---
# l1: 1 -> 2 -> 4, l2: 1 -> 3 -> 4
l1 = ListNode(1, ListNode(2, ListNode(4)))
l2 = ListNode(1, ListNode(3, ListNode(4)))
merged = merge_two_lists(l1, l2)
result = []
while merged:
    result.append(merged.val)
    merged = merged.next
print(f"Merged: {result}")  # [1, 1, 2, 3, 4, 4]

### Pattern 5 — In-place Modification

**Concept**: Modify node values or connections without allocating new nodes.

**Why it works**:
- **O(1) space**: Reuse existing nodes instead of creating new ones
- **Value copying**: Sometimes easier to copy values than restructure
- **Pointer rewiring**: Change connections, not values
- **Constraint exploitation**: Use problem constraints (e.g., "given only the node to delete")

**Common use cases**:
- Delete node without head pointer (LeetCode 237)
- Remove duplicates in sorted list (LeetCode 83)
- Swap adjacent nodes (LeetCode 24)
- Partition list (LeetCode 86)

**Key insight**: When you can't access previous nodes, think about copying from the next node or using clever pointer manipulation.

In [ ]:
# Delete node (LeetCode 237) - given only the node to delete
def delete_node(node):
    node.val = node.next.val  # copy next node's value
    node.next = node.next.next  # skip next node

# Swap nodes in pairs (LeetCode 24)
def swap_pairs(head):
    dummy = ListNode(0, head)
    prev = dummy
    
    while prev.next and prev.next.next:
        first = prev.next
        second = prev.next.next
        
        # Swap
        first.next = second.next
        second.next = first
        prev.next = second
        
        prev = first  # move to next pair
    
    return dummy.next

# --- Test ---
# Create: 1 -> 2 -> 3 -> 4
h = ListNode(1, ListNode(2, ListNode(3, ListNode(4))))
swapped = swap_pairs(h)
result = []
while swapped:
    result.append(swapped.val)
    swapped = swapped.next
print(f"Swapped pairs: {result}")  # [2, 1, 4, 3]

---

## Part 4: Common LeetCode Problems

### Problem 1 — LeetCode 206: Reverse Linked List

In [ ]:
# Iterative solution (preferred)
def reverse_list_iterative(head):
    prev = None
    curr = head
    while curr:
        nxt = curr.next
        curr.next = prev
        prev = curr
        curr = nxt
    return prev

# Recursive solution
def reverse_list_recursive(head):
    if not head or not head.next:
        return head
    new_head = reverse_list_recursive(head.next)
    head.next.next = head
    head.next = None
    return new_head

# --- Test ---
h = ListNode(1, ListNode(2, ListNode(3)))
rev = reverse_list_iterative(h)
result = []
while rev:
    result.append(rev.val)
    rev = rev.next
print(f"Reversed: {result}")  # [3, 2, 1]

### Problem 2 — LeetCode 21: Merge Two Sorted Lists

In [ ]:
# Already covered in Pattern 4, but here's a concise version
def merge_two_lists_concise(l1, l2):
    if not l1 or not l2:
        return l1 or l2
    
    if l1.val < l2.val:
        l1.next = merge_two_lists_concise(l1.next, l2)
        return l1
    else:
        l2.next = merge_two_lists_concise(l1, l2.next)
        return l2

# --- Test ---
l1 = ListNode(1, ListNode(3))
l2 = ListNode(2, ListNode(4))
merged = merge_two_lists_concise(l1, l2)
result = []
while merged:
    result.append(merged.val)
    merged = merged.next
print(f"Merged: {result}")  # [1, 2, 3, 4]

### Problem 3 — LeetCode 141: Linked List Cycle

In [ ]:
# Already covered in Pattern 1
# Alternative: Use set to track visited nodes (O(n) space)
def has_cycle_set(head):
    visited = set()
    while head:
        if head in visited:
            return True
        visited.add(head)
        head = head.next
    return False

# --- Test ---
# Create cycle: 1 -> 2 -> 3 -> 2
c1, c2, c3 = ListNode(1), ListNode(2), ListNode(3)
c1.next, c2.next, c3.next = c2, c3, c2
print(f"Has cycle (fast/slow): {has_cycle(c1)}")  # True
print(f"Has cycle (set): {has_cycle_set(c1)}")    # True

### Problem 4 — LeetCode 237: Delete Node in a Linked List

In [ ]:
# Already covered in Pattern 5
# Key insight: copy next node's value, then skip next node

# --- Test ---
# Create: 4 -> 5 -> 1 -> 9, delete node with value 5
h = ListNode(4, ListNode(5, ListNode(1, ListNode(9))))
node_to_delete = h.next  # node with value 5
delete_node(node_to_delete)
result = []
while h:
    result.append(h.val)
    h = h.next
print(f"After deletion: {result}")  # [4, 1, 9]

### Problem 5 — LeetCode 19: Remove Nth Node From End

In [ ]:
# Two pointers: advance fast by n steps, then move both
def remove_nth_from_end(head, n):
    dummy = ListNode(0, head)
    fast = dummy
    slow = dummy
    
    # Move fast n steps ahead
    for _ in range(n):
        fast = fast.next
    
    # Move both until fast reaches end
    while fast.next:
        fast = fast.next
        slow = slow.next
    
    # Remove node after slow
    slow.next = slow.next.next
    return dummy.next

# --- Test ---
# Create: 1 -> 2 -> 3 -> 4 -> 5, remove 2nd from end (value 4)
h = ListNode(1, ListNode(2, ListNode(3, ListNode(4, ListNode(5)))))
new_head = remove_nth_from_end(h, 2)
result = []
while new_head:
    result.append(new_head.val)
    new_head = new_head.next
print(f"After removal: {result}")  # [1, 2, 3, 5]

### Problem 6 — LeetCode 2: Add Two Numbers

In [ ]:
# Add numbers represented by linked lists (digits reversed)
def add_two_numbers(l1, l2):
    dummy = ListNode()
    curr = dummy
    carry = 0
    
    while l1 or l2 or carry:
        val1 = l1.val if l1 else 0
        val2 = l2.val if l2 else 0
        
        total = val1 + val2 + carry
        carry = total // 10
        curr.next = ListNode(total % 10)
        
        curr = curr.next
        if l1: l1 = l1.next
        if l2: l2 = l2.next
    
    return dummy.next

# --- Test ---
# 342 (2->4->3) + 465 (5->6->4) = 807 (7->0->8)
l1 = ListNode(2, ListNode(4, ListNode(3)))
l2 = ListNode(5, ListNode(6, ListNode(4)))
result = add_two_numbers(l1, l2)
output = []
while result:
    output.append(result.val)
    result = result.next
print(f"Sum: {output}")  # [7, 0, 8]

---

## Part 5: Pattern Summary

| Pattern | Description | Key Problems |
|---------|-------------|-------------|
| **Two Pointers** | Fast/slow for middle, cycle detection | 876, 141, 142 |
| **Dummy Head** | Simplify edge cases for head operations | 21, 203, 24 |
| **Reversal** | In-place reverse of entire/sublist | 206, 92, 25 |
| **Merge** | Combine sorted lists | 21, 23, 2 |
| **In-place** | Modify without extra space | 237, 24, 83 |
| **Recursion** | Elegant but uses stack space | 206, 21, 24 |

---

## Related LeetCode Problems

| Problem | Difficulty | Pattern |
|---------|------------|--------|
| [2. Add Two Numbers](https://leetcode.com/problems/add-two-numbers/) | Medium | Merge + Carry |
| [19. Remove Nth Node From End](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) | Medium | Two Pointers |
| [21. Merge Two Sorted Lists](https://leetcode.com/problems/merge-two-sorted-lists/) | Easy | Merge |
| [24. Swap Nodes in Pairs](https://leetcode.com/problems/swap-nodes-in-pairs/) | Medium | In-place |
| [25. Reverse Nodes in k-Group](https://leetcode.com/problems/reverse-nodes-in-k-group/) | Hard | Reversal |
| [83. Remove Duplicates from Sorted List](https://leetcode.com/problems/remove-duplicates-from-sorted-list/) | Easy | In-place |
| [92. Reverse Linked List II](https://leetcode.com/problems/reverse-linked-list-ii/) | Medium | Reversal |
| [141. Linked List Cycle](https://leetcode.com/problems/linked-list-cycle/) | Easy | Two Pointers |
| [142. Linked List Cycle II](https://leetcode.com/problems/linked-list-cycle-ii/) | Medium | Two Pointers |
| [160. Intersection of Two Linked Lists](https://leetcode.com/problems/intersection-of-two-linked-lists/) | Easy | Two Pointers |
| [203. Remove Linked List Elements](https://leetcode.com/problems/remove-linked-list-elements/) | Easy | Dummy Head |
| [206. Reverse Linked List](https://leetcode.com/problems/reverse-linked-list/) | Easy | Reversal |
| [237. Delete Node](https://leetcode.com/problems/delete-node-in-a-linked-list/) | Easy | In-place |

---

## Key Takeaways

| Concept | Key Point |
|---------|----------|
| **O(1) head ops** | Linked lists excel at front insertions/deletions |
| **Two pointers** | Fast/slow pattern solves many problems elegantly |
| **Dummy node** | Eliminates special cases for head operations |
| **In-place reversal** | Store `next`, reverse pointer, advance pointers |
| **Memory vs time** | Trade space for O(1) operations at boundaries |
| **When to use** | Frequent boundary ops vs random access needs |
| **Debugging** | Draw arrows to visualize pointer changes |
| **Edge cases** | Empty list, single node, last node |